# Plain SR-sentiment decodability check (D-032 follow-up)

The frozen factor screen rejected SR three-class sentiment because it did not beat
the **matched-wrong-EEG decoy** (correct$-$decoy $=-0.010$ macro-F1), even though it
**did** beat the ordinary metadata baseline (correct$-$metadata $=+0.020$). For a
3-class label the decoy (same-task, same-subject, *different-text* EEG) usually
shares the sentiment, so that control is unfair to the attribute. This check removes
the decoy and asks the simple question directly: **can a plain classifier read SR
sentiment off the frozen GLIM `correct` vector above chance?**

Same estimator family as the screen (standardized balanced L2 logistic regression),
subject-grouped CV, scored against analytic chance ($1/3$ balanced accuracy) **and** a
label-permutation empirical null. Uses the exact frozen vector artifact + rebuilt
protocol (hash-verified) and the pinned sklearn env, so it is apples-to-apples with
what failed. **CPU session is enough.** Attach `thestonedape/task-aware-eegtotext`
and `thestonedape/task-aware-eeg2text-frozen-glim-vectors`; Internet + `GITHUB_TOKEN`.
Held-out test is never touched.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'd08793b5df29e2a6b24b1322de9e0318e3eab776'
WORKTREE = '/kaggle/working/SemKey'
PACKAGES = '/kaggle/working/sr-decode-packages'
PROTOCOL = '/kaggle/working/frozen-recoverability-protocol'
OUTPUT = '/kaggle/working/sr-sentiment-decodability'
FACTOR_ID = 'sr_sentiment_3'
N_PERMUTATIONS, SEED = 200, 20260729
# Frozen artifact hashes (identical to run_frozen_factor_probes -> same vectors that failed).
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
EXPECTED_VECTOR_INDEX_SHA256 = '65d4a1f38f0df801f17ccb5504b4ee5d2f43aedcbc051e1ae1f405684ccad0e2'
EXPECTED_VECTOR_MANIFEST_SHA256 = '4861d9439a8a1253d87f38524a3720ea61d0934934589434f869b73263d36eba'
EXPECTED_REGISTRY_SHA256 = 'e4afa8a1bc859b86ad786016400035bec6d4a5280350abe271d0d74186af6517'
EXPECTED_RECOVERABILITY_ROWS_SHA256 = '4cdb7899088d32c37671bbd2088f650213ba001a771738a66a85660feee67f7d'
assert len(COMMIT) == 40 and all(len(v) == 64 for v in [
    EXPECTED_INDEX_SHA256, EXPECTED_VECTOR_INDEX_SHA256, EXPECTED_VECTOR_MANIFEST_SHA256,
    EXPECTED_REGISTRY_SHA256, EXPECTED_RECOVERABILITY_ROWS_SHA256])

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
if os.path.exists(PACKAGES): shutil.rmtree(PACKAGES)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                '--target', PACKAGES, 'numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2'], check=True)
PROBE_PYTHON = sys.executable
PROBE_ENV = os.environ.copy()
PROBE_ENV['PYTHONPATH'] = os.pathsep.join([PACKAGES, WORKTREE])
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as h:
    h.write("#!/usr/bin/env python3\nimport os, sys\np = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in p else 'x-access-token')\n")
os.chmod(askpass, 0o700)
cenv = os.environ.copy(); cenv.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE): shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=cenv)
finally:
    os.remove(askpass); del github_token, cenv
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
versions = json.loads(subprocess.check_output([PROBE_PYTHON, '-c', "import json, numpy, scipy, sklearn; print(json.dumps({'numpy': numpy.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__}))"], text=True, env=PROBE_ENV))
subprocess.run([PROBE_PYTHON, '-B', '-m', 'unittest',
                'evaluation.test_sentiment_decodability_check', 'evaluation.test_run_sr_decodability'],
               check=True, cwd=WORKTREE, env=PROBE_ENV)
print({'kernel_python': platform.python_version(), 'pinned_env': versions, 'self_tests': 'PASS'})

In [ ]:
def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as h:
        for block in iter(lambda: h.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()
shard_manifests = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(shard_manifests) == 1, ('Attach exactly one canonical sharded dataset', shard_manifests)
dataset_root = os.path.dirname(os.path.dirname(shard_manifests[0]))
vector_manifests = glob.glob('/kaggle/input/**/vector_manifest.json', recursive=True)
assert len(vector_manifests) == 1, ('Attach exactly one frozen GLIM vector dataset', vector_manifests)
vector_root = os.path.dirname(vector_manifests[0])
assert digest(vector_manifests[0]) == EXPECTED_VECTOR_MANIFEST_SHA256, 'vector manifest hash mismatch'
assert digest(os.path.join(vector_root, 'vector_index.csv')) == EXPECTED_VECTOR_INDEX_SHA256, 'vector index hash mismatch'
print({'dataset_root': dataset_root, 'vector_root': vector_root})

In [ ]:
if os.path.exists(PROTOCOL): shutil.rmtree(PROTOCOL)
subprocess.run([PROBE_PYTHON, os.path.join(WORKTREE, 'evaluation', 'build_recoverability_protocol.py'),
                '--dataset-root', dataset_root, '--output-root', PROTOCOL,
                '--expected-index-sha256', EXPECTED_INDEX_SHA256], check=True, env=PROBE_ENV)
assert digest(os.path.join(PROTOCOL, 'recoverability_registry.json')) == EXPECTED_REGISTRY_SHA256, 'registry hash mismatch'
assert digest(os.path.join(PROTOCOL, 'recoverability_rows.csv')) == EXPECTED_RECOVERABILITY_ROWS_SHA256, 'rows hash mismatch'
print({'protocol': 'PASS (byte-identical to the frozen factor-screen protocol)'})

In [ ]:
if os.path.exists(OUTPUT): shutil.rmtree(OUTPUT)
proc = subprocess.run([PROBE_PYTHON, os.path.join(WORKTREE, 'evaluation', 'run_sr_decodability.py'),
                       '--vector-root', vector_root, '--protocol-root', PROTOCOL, '--output-root', OUTPUT,
                       '--factor-id', FACTOR_ID, '--n-permutations', str(N_PERMUTATIONS), '--seed', str(SEED),
                       '--expected-index-sha256', EXPECTED_INDEX_SHA256,
                       '--expected-vector-index-sha256', EXPECTED_VECTOR_INDEX_SHA256],
                      check=True, capture_output=True, text=True, env=PROBE_ENV)
result = json.loads(proc.stdout)

In [ ]:
r = result
print('=== SR THREE-CLASS SENTIMENT: PLAIN DECODABILITY (decoy control removed) ===')
print('trials / subjects / classes :', r['n'], '/', r['n_groups'], '/', r['n_classes'], '| class counts', r['class_counts'])
print('CV folds used               :', r['n_splits_used'])
print('balanced accuracy (real)    :', r['real']['balanced_accuracy'])
print('  analytic chance           :', r['analytic_chance_balanced_accuracy'], '(1/3)')
print('  permutation chance +- sd  :', r['permutation_chance_balanced_accuracy'], '+-', r['permutation_chance_sd'])
print('  permutation p-value       :', r['p_value_permutation'])
print('accuracy / macro-F1 (real)  :', r['real']['accuracy'], '/', r['real']['macro_f1'])
print('report sha256               :', r['report_sha256'])
print('VERDICT                     :', r['verdict'])

Save `sr_sentiment_decodability.json` for provenance. Interpretation:

- **DECODABLE** (balanced accuracy clearly above the permutation chance, p < 0.05) ->
  the sentiment signal *is* in the frozen GLIM vector; the factor screen's rejection
  was driven by the identity-tuned matched-wrong decoy, not by absence of signal.
  We then report the SR null honestly as a *control-design* artifact, and NR/SR
  attribute decodability becomes a real (if modest) positive to note -- not a new
  research program, just an honest correction.
- **AT CHANCE** (balanced accuracy ~ permutation chance, p not significant) -> the
  signal genuinely is not recoverable from this frozen, retrieval-tuned
  representation; the original null stands and "we tested it wrong" is ruled out.

Either way this is a self-contained diagnostic; it does not reopen the task-aware
program (D-032) and the held-out test stays sealed.